# IC-SHM 2026 Project 2 — End-to-End 3D Reconstruction & Baseline Semantic Fusion

This notebook walks through the entire pipeline from scratch on the real contest dataset:
1. **Step 1: 3D Triangulation via pycolmap**: Loads calibrated camera poses and triangulates 3D points via LO-RANSAC.
2. **Step 2: Visualize Raw Spatial 3D Point Cloud**: Plots the unclassified 3D bridge structure and UAV camera trajectory.
3. **Step 3: Baseline Semantic Fusion (Naive Plurality Voting)**: Implements standard unconstrained majority voting without slender-structure safeguards.
4. **Step 4: Visualize Baseline 3D Semantic Point Cloud**: Interactively inspects the baseline result and reveals challenges (floating noise, cable thinning).
5. **Step 5: Baseline Quantitative Benchmark**: Computes baseline $mIoU$, $IoU_{cable}$, Deck MAD, and Cable Fan Deviation.

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
from collections import Counter
import plotly.graph_objects as go
import plotly.express as px
from PIL import Image

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.reconstruction.pycolmap_reconstructor import PycolmapReconstructor, load_contest_model
from src.reconstruction.semantic_projector import CLASS_NAMES, CLASS_COLORS
from src.reconstruction.visualizer import CLASS_HEX_COLORS, create_interactive_3d_figure
from src.evaluation.metrics import evaluate_predictions, evaluate_multiview_holdout
from IPython.display import display, Markdown

DATASET_DIR = '../data/Contest Dataset'
COLMAP_DIR = os.path.join(DATASET_DIR, 'camera_parameters')
MASKS_DIR = '../outputs/gt_masks'
print('✅ Environment and modules loaded successfully!')

## 1. Step 1: 3D Triangulation from COLMAP Parameters

We load the 400 camera poses (`cameras.txt`, `images.txt`) and triangulate the 86,336 feature tracks using LO-RANSAC.

In [ ]:
reconstructor = PycolmapReconstructor(COLMAP_DIR)
camera, images, pts3d = reconstructor.load()

errors = np.array([p.error for p in reconstructor.reconstruction.points3D.values()])
mean_reproj_err = float(errors.mean())
median_reproj_err = float(np.median(errors))

print(f"\n📊 Triangulation Summary:")
print(f"  • Total Registered Images: {len(images):,}")
print(f"  • Total Triangulated 3D Points: {len(pts3d):,}")
print(f"  • Mean Reprojection Error: {mean_reproj_err:.2f} px")
print(f"  • Median Reprojection Error: {median_reproj_err:.2f} px")

## 2. Step 2: Visualize Raw Spatial 3D Point Cloud

We extract the spatial $(X, Y, Z)$ coordinates of all triangulated points and render the 3D bridge structure colored by height/depth.

In [ ]:
xyz_array = np.array([pt.xyz for pt in pts3d.values()])
cam_centers = np.array([-img.R.T @ img.T.flatten() for img in images.values()])

# Subsample points for smooth interactive rendering
sample_rate = 2
sampled_xyz = xyz_array[::sample_rate]

fig_raw = go.Figure()
# 3D Bridge Points
fig_raw.add_trace(go.Scatter3d(
    x=sampled_xyz[:, 0], y=sampled_xyz[:, 1], z=sampled_xyz[:, 2],
    mode='markers',
    marker=dict(size=1.8, color=sampled_xyz[:, 2], colorscale='Viridis', opacity=0.8),
    name='3D Triangulated Points'
))
# UAV Camera Flight Trajectory
fig_raw.add_trace(go.Scatter3d(
    x=cam_centers[:, 0], y=cam_centers[:, 1], z=cam_centers[:, 2],
    mode='markers+lines',
    marker=dict(size=3, color='red'),
    line=dict(color='orange', width=1.5),
    name='Drone Camera Trajectory'
))

fig_raw.update_layout(
    title=f'Raw 3D Point Cloud ({len(xyz_array):,} points) & 400 Drone Camera Centers',
    template='plotly_dark',
    scene=dict(aspectmode='data'),
    height=650,
)
fig_raw.show()

## 3. Step 3: Baseline Semantic Fusion (Naive Plurality Voting)

### What is "Baseline Semantic Fusion"?
- **Mechanism**: For every 3D point $X_i$, we retrieve all 2D observation coordinates $(u, v)$ across observing camera frames, sample the 2D mask pixel labels, and take the standard **mode (argmax count)**:
  $$\hat{c}_{\text{base}} = \arg\max_{c \in \{0,1,2,3,4\}} \text{Count}(c)$$
- **No slender-structure safeguards**: No $>50\%$ strict threshold for stay cables.
- **No tie-breaking priority**: Defaults to arbitrary first max count.
- **No geometric filtering**: Raw point cloud as directly produced by voting.

In [ ]:
# 1. Preload 2D Masks into memory cache
print('🔄 Preloading 2D GT masks into cache...')
mask_cache = {}
for img_id, img_pose in images.items():
    mask_name = os.path.splitext(img_pose.name)[0] + '.png'
    mask_path = os.path.join(MASKS_DIR, mask_name)
    if os.path.exists(mask_path):
        mask_cache[img_id] = np.array(Image.open(mask_path))

print(f'✅ Loaded {len(mask_cache)} masks into memory.')

# 2. Naive Plurality Voting Function (Baseline)
def naive_plurality_vote(labels):
    if not labels:
        return 0
    counts = Counter(labels)
    # Standard mode: pick class with highest vote count
    return counts.most_common(1)[0][0]

# 3. Run Baseline Semantic Projection across all 3D points
t0 = time.time()
baseline_classes = {}
baseline_colors = {}
class_counts = Counter()

for p3d_id, pt3d in pts3d.items():
    observed_labels = []
    for img_id, pt2d_idx in zip(pt3d.image_ids, pt3d.point2d_idxs):
        if img_id not in mask_cache:
            continue
        img_pose = images[img_id]
        mask = mask_cache[img_id]
        u, v, _ = img_pose.points2d[pt2d_idx]
        x, y = int(round(u)), int(round(v))
        h, w = mask.shape
        if 0 <= x < w and 0 <= y < h:
            observed_labels.append(int(mask[y, x]))
            
    final_class = naive_plurality_vote(observed_labels)
    baseline_classes[p3d_id] = final_class
    baseline_colors[p3d_id] = CLASS_COLORS.get(final_class, CLASS_COLORS[0])
    class_counts[final_class] += 1

t1 = time.time()
print(f'✅ Baseline Semantic Fusion completed in {t1 - t0:.2f} seconds!')

# Display Class Breakdown
df_baseline = pd.DataFrame([
    {'Class ID': cid, 'Class Name': CLASS_NAMES[cid], 'Point Count': class_counts[cid], 
     'Percentage (%)': round(class_counts[cid] / len(pts3d) * 100, 2)}
    for cid in sorted(CLASS_NAMES.keys())
])
display(df_baseline)

## 4. Step 4: Visualize Baseline 3D Semantic Point Cloud

We visualize the raw baseline point cloud where colors represent semantic classes:
- 🔴 `deck` — Red `[255, 0, 0]`
- 🔵 `stay_cable` — Cyan `[0, 255, 255]`
- 🟢 `tower` — Green `[0, 255, 0]`
- 🟡 `foundation` — Yellow `[255, 255, 0]`
- ⚪ `background` — Gray `[128, 128, 128]`

In [ ]:
base_cids = np.array(list(baseline_classes.values()))
base_rgb = np.array(list(baseline_colors.values()))
base_xyz = np.array([pt.xyz for pt in pts3d.values()])

# Render Baseline 3D Interactive Figure
fig_base_3d = create_interactive_3d_figure(base_xyz, base_rgb, base_cids, point_size=2.2)
fig_base_3d.update_layout(
    title=f'Baseline 3D Semantic Point Cloud ({len(base_xyz):,} points — Unfiltered Naive Fusion)',
    height=700
)
fig_base_3d.show()

## 5. Step 5: Quantitative Evaluation of Baseline Model

We compute the benchmark metrics on the raw baseline point cloud to quantify baseline performance before applying our proposed structure-aware filtering pipeline.

In [ ]:
# Run Authentic 80/20 Multi-View Hold-Out Cross-Validation on Baseline Voting
report_base = evaluate_multiview_holdout(
    pts3d=pts3d,
    images=images,
    mask_cache=mask_cache,
    holdout_ratio=0.20,
    seed=42,
    vote_func=naive_plurality_vote,
    lateral_axis=np.array([0.0, 0.0, 1.0]),
    d_left=-2.15,
    d_right=2.15,
    mean_reproj_error=mean_reproj_err,
    estimated_bridge_area=1200.0
)

display(Markdown(report_base.to_markdown()))
print('\n💡 Analysis of Baseline Limitations:')
print(f"  • Baseline Stay-Cable IoU is only {report_base.ious.get(2, 0)*100:.2f}% (due to 2D background bleeding and naive voting).")
print(f"  • Baseline Structural mIoU is {report_base.miou_structural*100:.2f}%.")
print('  • In our proposed method, the >50% cable safeguard + 8-stage geometric filter lifts this benchmark significantly!')